# conflibertR: end-to-end walkthrough

[`conflibertR`](https://github.com/shreyasmeher/conflibertR) is an R interface to **ConfliBERT**, a transformer model purpose-built for conflict and political-violence text. This notebook walks through every major capability of the package and ends with an extended active-learning workflow.

**Sections**

1. Setup (Colab or local)
2. Named Entity Recognition
3. Binary Classification
4. Multilabel Event Classification
5. Question Answering
6. Benchmarking the pretrained classifier
7. Fine-tuning a custom classifier
8. Comparing multiple base models
9. Active Learning (the main event)
    - Sessions, query batches, and labeling
    - Shiny gadget vs. console-only labeling
    - Query strategies (entropy / margin / least confidence)
    - Diversity-aware batches
    - LoRA adapters
    - Saving and reloading the trained model

> **Kernel:** R (via [IRkernel](https://github.com/IRkernel/IRkernel)).
>
> - **On Colab:** *Runtime → Change runtime type → R*, and pick a GPU accelerator (T4 is fine) for fine-tuning + active learning. Then run the **Setup** cells below in order, restart the runtime when prompted, and pick up at the *Post-restart* cell.
> - **Locally:** `install.packages('IRkernel'); IRkernel::installspec()` once, then run the same setup cells (the `system('apt-get …')` cell is Colab-only — skip it).

> **GPU note:** Fine-tuning and active-learning rounds are much faster on GPU but will run on CPU. Inference (NER / classify / multilabel / QA) is fine on CPU.

## 1. Setup

Three steps:

1. Install OS dev libs (Colab only — needed so R can build packages with `curl` / `xml2` / `openssl` deps).
2. Install the R package from GitHub.
3. Install the Python backend.
   - **On Colab** we skip `conflibert_install()` and install the missing Python packages directly into Colab's bundled Python (its `/usr/local/bin/python` can't create a venv, and torch + tensorflow already ship in the runtime — so we only pull `transformers`, `accelerate`, `peft`, `scikit-learn`, `tf-keras`).
   - **Locally**, `conflibert_install(method = 'auto')` picks conda if available, else virtualenv.

After Step 3, **restart the runtime** and pick up at the *Post-restart* cell. Together this is ~2–3 min on Colab (~5–10 min locally if torch has to download).

In [ ]:
## Step 1 — Colab only. Skip locally.
## Installs OS dev libs that some R packages link against.
if (file.exists('/etc/lsb-release')) {
  cmd <- 'apt-get update -qq && apt-get install -y --no-install-recommends libcurl4-openssl-dev libssl-dev libxml2-dev'
  status <- system(cmd)
  if (status != 0) {
    message('apt-get returned non-zero (', status, '). On Colab the libs are usually already present — try Step 2 anyway.')
  }
}

In [ ]:
# Step 2 — R package from GitHub
if (!requireNamespace('remotes', quietly = TRUE)) install.packages('remotes')
remotes::install_github('shreyasmeher/conflibertR', upgrade = 'never')

In [ ]:
## Step 3 — Python backend.
##
## On Colab, /usr/local/bin/python can't create a venv (no ensurepip),
## so we install deps directly into the system Python. Torch + tensorflow
## already ship with Colab; we only need the missing extras.
##
## Locally with conda available, we use the normal conflibert_install path.
on_colab <- file.exists('/etc/lsb-release') && file.exists('/usr/local/bin/python')

if (on_colab) {
  system('pip install -q "transformers>=4.40,<5" accelerate peft scikit-learn tf-keras')
} else {
  conflibertR::conflibert_install(method = 'auto')
}

### ⚠️ Restart the runtime now

*Runtime → Restart session* (Colab) or restart your R kernel (local). This is required so `conflibertR` picks up the new virtualenv on its next load. Then continue from the **Post-restart** cell below — you do **not** need to re-run the install cells above.

### Post-restart

Load the package and confirm the Python backend is wired up. `conflibert_models()` should return a tibble of available base models without errors.

In [ ]:
## On Colab we skipped the venv and installed into the system Python,
## so point reticulate at it before loading the package.
if (file.exists('/etc/lsb-release') && file.exists('/usr/local/bin/python')) {
  Sys.setenv(RETICULATE_PYTHON = '/usr/local/bin/python')
}

library(conflibertR)
conflibert_models()

## 2. Named Entity Recognition

Extract persons, organisations, locations, weapons, and temporal expressions from conflict-related text. Pass a single string or a vector — the function is vectorized.

*The first inference call downloads the model from HuggingFace (~400 MB). Subsequent calls reuse the cached weights.*

In [ ]:
conflibert_ner(
  'The United Nations deployed peacekeepers to eastern Congo after rebel forces attacked Goma on Tuesday.'
)

In [ ]:
conflibert_ner(c(
  'NATO forces launched airstrikes near Tripoli in March 2024.',
  'President Zelenskyy met with General Syrskyi in Kyiv.'
))

## 3. Binary Classification

Is this text describing a conflict event? Each row gets a label, numeric class, confidence, and per-class probabilities.

In [ ]:
conflibert_classify(c(
  'Armed militants stormed a military outpost killing twelve soldiers.',
  'The European Central Bank held interest rates steady at 4.5 percent.',
  'Protesters clashed with riot police outside the parliament building.',
  'A new trade agreement was signed between Japan and Australia.'
))

## 4. Multilabel Event Classification

Score text against four event categories — *Armed Assault*, *Bombing or Explosion*, *Kidnapping*, *Other* — independently. A single text can be flagged for several at once.

In [ ]:
conflibert_multilabel(c(
  'A car bomb exploded near the government ministry in Baghdad.',
  'Gunmen kidnapped three aid workers travelling through the Sahel region.'
))

## 5. Question Answering

Extractive QA — pull spans directly out of a passage.

In [ ]:
context <- 'On 15 September 2023, ethnic clashes erupted in Manipur, India.
  The Indian Army deployed two battalions to restore order.
  At least 60 people were killed and over 200 injured in the violence.'

conflibert_qa(context, 'How many people were killed?')
conflibert_qa(context, 'Who was deployed to restore order?')
conflibert_qa(context, 'Where did the clashes occur?')

## 6. Benchmarking the pretrained classifier

Evaluate the pretrained ConfliBERT binary classifier on labeled data. The package ships small example datasets you can use to get started.

In [ ]:
data <- conflibert_example('binary')
str(data, max.level = 2)

In [ ]:
bench <- conflibert_benchmark(data$test$text, data$test$label)
bench

## 7. Fine-tuning a custom classifier

Train your own binary classifier on labeled data. The bundled example has 80 train / 20 dev / 20 test examples.

*This is the first cell that really wants a GPU. On a T4 it finishes in under a minute; on CPU it can take 5–10 minutes.*

In [ ]:
result <- conflibert_finetune(
  train  = data$train,
  dev    = data$dev,
  test   = data$test,
  model  = 'ConfliBERT',
  task   = 'binary',
  epochs = 3
)

cat('Accuracy:', result$metrics$accuracy, '\n')
cat('F1 Score:', result$metrics$f1, '\n')

In [ ]:
head(result$predictions)
head(result$probabilities)
result$model_dir   # NULL unless save_dir was set

### Multiclass fine-tuning

The bundled multiclass dataset has four event types: *Diplomacy*, *Armed Conflict*, *Protest*, *Humanitarian*.

In [ ]:
mc_data <- conflibert_example('multiclass')

mc_result <- conflibert_finetune(
  train  = mc_data$train,
  dev    = mc_data$dev,
  test   = mc_data$test,
  model  = 'ConfliBERT',
  task   = 'multiclass',
  epochs = 3
)

cat('Multiclass Accuracy:', mc_result$metrics$accuracy, '\n')
cat('Multiclass F1:      ', mc_result$metrics$f1, '\n')

## 8. Comparing multiple base models

Pit ConfliBERT against general-purpose transformers on the same task. `conflibert_compare()` fine-tunes each one and returns a tidy comparison table.

*This trains three models in sequence — expect ~3× the time of the single fine-tune above.*

In [ ]:
comparison <- conflibert_compare(
  train  = data$train,
  dev    = data$dev,
  test   = data$test,
  models = c('ConfliBERT', 'BERT Base Uncased', 'DistilBERT Base'),
  task   = 'binary',
  epochs = 3
)
comparison

## 9. Active Learning

Labeling text is expensive. Active learning lets you spend annotation effort on the examples a model is *most uncertain* about, so each label has maximum impact on performance.

The conflibertR loop:

1. Start from a tiny labeled seed and an unlabeled pool.
2. Train a model on the seed; the package returns the most uncertain samples from the pool.
3. Label those samples, submit them, and the model retrains and picks the next uncertain batch.
4. Repeat until the pool is exhausted or metrics plateau.
5. Save the final model as a HuggingFace checkpoint.

The whole loop is three functions: `conflibert_active_start()`, `conflibert_active_next()`, `conflibert_active_save()`. A fourth — `conflibert_active_label()` — opens an interactive labeling UI (skip it on Colab; we use the oracle below).

### 9.1 Example data

The package bundles a small demo dataset: a 20-text labeled seed, a 61-text unlabeled pool, and a dev set. It also includes oracle labels for the pool so we can simulate a full loop without a human in the loop. **In a real workflow you'd label `session$query` yourself; the oracle is for testing only.**

In [ ]:
demo <- conflibert_example('active')

cat('Seed (labeled): ', nrow(demo$seed), '\n')
cat('Pool (unlabeled):', length(demo$pool), '\n')
cat('Dev:             ', nrow(demo$dev), '\n')
cat('Oracle labels:   ', length(demo$pool_labels), '(simulation only)\n')

head(demo$seed)

### 9.2 Starting a session

`conflibert_active_start()` trains a classifier on the seed and returns a session object containing the first uncertain batch.

In [ ]:
session <- conflibert_active_start(
  seed       = demo$seed,
  pool       = demo$pool,
  dev        = demo$dev,
  model      = 'ConfliBERT',
  task       = 'binary',
  strategy   = 'entropy',     # or 'margin', 'least_confidence'
  query_size = 10,
  epochs     = 1
)

session

What's in a session:

- `session$query` — tibble of texts to label next, with an `uncertainty` column.
- `session$metrics` — per-round dev metrics (accuracy, F1) tracked across rounds.
- `session$labeled_n` / `session$pool_n` — progress counters.
- `session$done` — `TRUE` once the pool is exhausted.

In [ ]:
session$query

In [ ]:
session$metrics

### 9.3 Labeling and iterating

**Option A — Shiny gadget (real labeling, local R).** Opens a modal dialog with radio buttons for each class. Skip on Colab — the gadget needs an interactive desktop session.

```r
labels  <- conflibert_active_label(session)
session <- conflibert_active_next(session, labels = labels)
```

**Option B — by hand.** Provide a vector in the same order as `session$query`:

```r
labels  <- c(1, 0, 1, 0, 0, 1, 0, 1, 0, 1)
session <- conflibert_active_next(session, labels = labels)
```

**Option C — oracle simulation (this notebook).** We use the bundled oracle so the notebook runs end-to-end on Colab without any human labeling:

In [ ]:
labels  <- unname(demo$pool_labels[session$query$text])
session <- conflibert_active_next(session, labels = labels)
session

Now run a few more rounds. Each round trains on the growing labeled set and queries a fresh uncertain batch. We stop early if the pool is exhausted (`session$done`).

In [ ]:
for (round in 2:5) {
  if (session$done) break
  labels  <- unname(demo$pool_labels[session$query$text])
  session <- conflibert_active_next(session, labels = labels)
  cat(sprintf('Round %d  labeled=%d  pool=%d\n',
              round, session$labeled_n, session$pool_n))
}

session$metrics

### 9.4 Visualizing progress

`plot()` produces a two-panel diagnostic: the learning curve on top (metrics vs. labeled-set size) and the query uncertainty trend on the bottom. When mean uncertainty flattens, the model is no longer finding informative samples — a good signal to stop labeling.

In [ ]:
plot(session)

# Or a single panel:
# plot(session, which = 'metrics')
# plot(session, which = 'uncertainty')

### 9.5 Query strategies

Three uncertainty strategies, picked at `conflibert_active_start()` time via `strategy =`:

| Strategy            | Picks samples with…                                 | When to use                          |
|---------------------|------------------------------------------------------|--------------------------------------|
| `entropy` (default) | highest Shannon entropy of class probs              | binary or multiclass; safe default   |
| `margin`            | smallest gap between top-two class probs            | targets decision-boundary cases      |
| `least_confidence`  | lowest max class prob                               | simplest baseline                    |

Restart with a different strategy whenever you like:

In [ ]:
session_margin <- conflibert_active_start(
  seed = demo$seed, pool = demo$pool, dev = demo$dev,
  strategy = 'margin', query_size = 10, epochs = 1
)
head(session_margin$query)

### 9.6 Diversity-aware batches

Pure uncertainty sampling can pick several near-duplicates in one batch — wasteful when your pool has many similar texts. Pass `diverse = TRUE` to cluster the top-scoring candidates in the model's embedding space and pick the highest-scoring sample from each cluster.

In [ ]:
session_div <- conflibert_active_start(
  seed = demo$seed, pool = demo$pool, dev = demo$dev,
  strategy             = 'entropy',
  diverse              = TRUE,
  diversity_candidates = 30,   # defaults to 3 * query_size
  query_size           = 10,
  epochs               = 1
)
head(session_div$query)

### 9.7 LoRA fine-tuning

For bigger base models or tighter GPU budgets, train only a low-rank adapter each round. The adapter is merged into the base model before every round ends, so scoring, saving, and reloading behave exactly like full fine-tuning.

In [ ]:
session_lora <- conflibert_active_start(
  seed = demo$seed, pool = demo$pool, dev = demo$dev,
  model      = 'DeBERTa v3 Base',
  use_lora   = TRUE,
  lora_rank  = 8,
  lora_alpha = 16,
  query_size = 10,
  epochs     = 1
)
session_lora

### 9.8 Saving and reloading the model

Persist the final model as a standard HuggingFace checkpoint. On Colab, save into `/content` and download it (or push to the HuggingFace Hub) before the runtime is recycled.

In [ ]:
save_dir <- if (file.exists('/etc/lsb-release')) '/content/my_al_model' else 'my_al_model'
conflibert_active_save(session, save_dir)
list.files(save_dir)

Reload it from any `transformers` tool — for example from Python:

```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model = AutoModelForSequenceClassification.from_pretrained('/content/my_al_model')
tok   = AutoTokenizer.from_pretrained('/content/my_al_model')
```

## Tips

- **Start small.** A seed of 10–50 texts is often enough; active learning shines when the pool is much larger than the labeled set.
- **Watch uncertainty trends, not just metrics.** If dev metrics are noisy on small sets, the uncertainty panel often tells a clearer story about whether the model is still learning.
- **Parameters carry across rounds.** Model, task, strategy, query size, and training hyperparameters are fixed at `conflibert_active_start()` time and reused for every subsequent round.
- **Sessions are in-memory.** The trained model lives inside the session object as a Python handle. `saveRDS()` won't serialize it — use `conflibert_active_save()` to persist, and re-run rounds from a fresh session if needed.
- **Colab runtimes are ephemeral.** Anything in `/content` disappears when the runtime is recycled. Download saved models or push them to the Hub.

## Function reference

| Function                       | Task                                       |
|--------------------------------|--------------------------------------------|
| `conflibert_ner()`             | Named entity recognition                   |
| `conflibert_classify()`        | Binary conflict classification             |
| `conflibert_multilabel()`      | Multilabel event type scoring              |
| `conflibert_qa()`              | Extractive question answering              |
| `conflibert_benchmark()`       | Evaluate the pretrained classifier         |
| `conflibert_finetune()`        | Train a custom classifier                  |
| `conflibert_compare()`         | Compare multiple model architectures       |
| `conflibert_example()`         | Load bundled example datasets              |
| `conflibert_models()`          | List available base models                 |
| `conflibert_active_start()`    | Start an active-learning session           |
| `conflibert_active_next()`     | Submit labels, get next query batch        |
| `conflibert_active_label()`    | Shiny gadget for interactive labeling      |
| `conflibert_active_save()`     | Save the final model as HF checkpoint      |

Reference: Brandt et al. (2025), *Extractive versus Generative Language Models for Political Conflict Text Classification*, **Political Analysis**.